# 3D-ST — Plantilla de Google Colab

Esta plantilla general permite cargar una foto y generar un modelo 3D con TripoSR. Si llegaste desde la web de 3D-ST y descargaste el cuaderno personalizado, la foto ya está incorporada y no necesitas cargarla otra vez.

## 1. Seleccionar fotografía

Esta celda es para usar la plantilla manualmente. En el cuaderno personalizado generado por la web de 3D-ST, puedes omitirla porque la imagen ya viene incluida.

In [ ]:
from google.colab import files
from pathlib import Path

cargas = files.upload()
if not cargas:
    raise RuntimeError('Carga una fotografía PNG, JPG, JPEG o WebP.')
nombre = next(iter(cargas))
imagen = Path('/content') / nombre
print('Fotografía seleccionada:', imagen)

## 2. Preparar entorno

Si está disponible, activa una GPU desde Entorno de ejecución → Cambiar tipo de entorno de ejecución. La instalación puede tardar varios minutos.

In [ ]:
import os, pathlib, subprocess, sys
def ejecutar(cmd):
    print('$', ' '.join(cmd))
    subprocess.run(cmd, check=True)

# Colab administra su propio Python (en esta sesión puede ser 3.13).
# No creamos un entorno virtual, porque algunas imágenes de Colab no incluyen
# ensurepip/venv. Se conserva el PyTorch y CUDA que Colab ya tiene instalados.
PY = sys.executable
print('Python de Colab:', sys.version)
try:
    import torch
    print('PyTorch preinstalado:', torch.__version__)
    print('CUDA disponible:', torch.cuda.is_available())
except Exception as e:
    raise RuntimeError('No se pudo importar PyTorch del entorno de Colab: ' + str(e))

ejecutar([PY, '-m', 'pip', 'install', '--upgrade', 'scikit-build-core', 'pybind11', 'ninja', 'cmake'])
ejecutar([PY, '-m', 'pip', 'install', '--no-cache-dir', '--no-build-isolation', 'git+https://github.com/tatsy/torchmcubes.git'])
ejecutar([PY, '-m', 'pip', 'install', 'omegaconf==2.3.0', 'Pillow', 'einops==0.7.0', 'transformers==4.35.0', 'trimesh==4.0.5', 'rembg', 'onnxruntime', 'huggingface-hub', 'imageio[ffmpeg]', 'gradio', 'xatlas', 'moderngl==5.10.0'])
if not pathlib.Path('/content/TripoSR/run.py').exists():
    ejecutar(['git', 'clone', 'https://github.com/VAST-AI-Research/TripoSR.git', '/content/TripoSR'])
ejecutar([PY, '-c', "import torch, torchmcubes, rembg, onnxruntime; print('PyTorch:', torch.__version__); print('CUDA:', torch.version.cuda); print('GPU disponible:', torch.cuda.is_available()); print('Dependencias: OK')"])
try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception:
    print('No se detectó una GPU NVIDIA. Si Colab ofrece GPU, actívala en Entorno de ejecución → Cambiar tipo de entorno de ejecución.')
print('Entorno preparado.')


## 3. Generar el modelo 3D

TripoSR estima las partes no visibles. El resultado puede necesitar reparación antes de imprimirlo en 3D.

In [ ]:
import pathlib, subprocess, os
import sys
PY = sys.executable
SALIDA = '/content/3dst-output'
os.makedirs(SALIDA, exist_ok=True)
subprocess.run([PY, '/content/TripoSR/run.py', str(imagen), '--output-dir', SALIDA, '--model-save-format', 'glb'], check=True)
archivos = [p for p in pathlib.Path(SALIDA).rglob('*') if p.is_file()]
if not archivos:
    raise FileNotFoundError('No se encontraron archivos de salida.')
for archivo in archivos:
    print(archivo, f'({archivo.stat().st_size / 1024:.1f} KB)')


## 4. Descargar el modelo 3D

In [ ]:
from google.colab import files
import pathlib
salida = pathlib.Path('/content/3dst-output')
modelos = list(salida.rglob('*.glb')) + list(salida.rglob('*.obj'))
if not modelos:
    raise FileNotFoundError('No se encontró ningún archivo GLB u OBJ.')
for modelo in modelos:
    print('Descargando:', modelo.name)
    files.download(str(modelo))

**Importante:** Colab puede no ofrecer GPU en todas las sesiones. El modelo es una estimación visual, no una reproducción dimensional exacta.